In [3]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool, GCNConv, GATConv, SAGEConv
import torch.nn as nn
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, GATConv, ChebConv, SAGEConv, global_mean_pool
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
import numpy as np
import mne
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool
import torch_geometric
from sklearn.model_selection import GroupKFold,LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/typing.py:31: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: dlopen(/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so, 0x0006): Symbol not found: __ZN2at4_ops6narrow4callERKNS_6TensorExxx
  Referenced from: <FC4F5CE6-3038-3A9A-B98B-661CD724F01B> /Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so
  Expected in:     <66FB8649-BB87-3CD6-A177-462038DCAE02> /Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/typing.py:42: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: dlopen(/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so, 0x

Load and preprocess data

In [4]:
# This function loads the gdf file and applies band pass filter to it and returns the features and labels
def read_data(path):
  raw = mne.io.read_raw_gdf(path,eog=['EOG-left', 'EOG-central', 'EOG-right'],preload=True)
  raw.drop_channels(['EOG-left', 'EOG-central', 'EOG-right'])
  raw.set_eeg_reference()
  raw.filter(l_freq=8., h_freq=30.)
  events = mne.events_from_annotations(raw)
  epoch = mne.Epochs(raw,events[0],event_id=[7,8,9,10],tmin = -0.1,tmax=0.7,on_missing='warn')
  labels= epoch.events[:,-1] 
  features= epoch.get_data()
  sfreq = raw.info['sfreq']
  return features,labels,sfreq

Shape of the features after using this function will be Events X channelss X datapoints

In [5]:
data, labels, sfreq= read_data('/Users/siddharth/Downloads/BCICIV_2a_gdf/A01T.gdf')

Extracting EDF parameters from /Users/siddharth/Downloads/BCICIV_2a_gdf/A01T.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/Users/siddharth/opt/anaconda3/envs/Brainconnectivity/lib/python3.9/contextlib.py:126: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 s)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.1s remaining:    0.0s


Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '769', '770', '771', '772']
Not setting metadata
288 matching events found
Setting baseline interval to [-0.1, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 288 events and 201 original time points ...
0 bad epochs dropped


[Parallel(n_jobs=1)]: Done  22 out of  22 | elapsed:    0.3s finished


### NODE FEATURE EXTRACTION

In [4]:
labels = labels - 7

In [5]:
from collections import Counter
labels_distribution = Counter(labels)

In [7]:
from matplotlib import pyplot as plt

Counter({3: 72, 2: 72, 1: 72, 0: 72})

In [9]:
def calculate_band_power(data,sfreq):
  band_powers = []
  
  for epoch in data:  # Loop over each epoch
      epoch_powers = []
      for channel_data in epoch:  # Loop over each channel in the epoch
          freqs, psd = welch(channel_data, sfreq, nperseg=sfreq*2)  # Compute the PSD
          # Compute the band power by integrating the power spectral density (PSD) in the 8-30 Hz band
          band_power = np.trapz(psd[(freqs >= 8) & (freqs <= 30)], freqs[(freqs >= 8) & (freqs <= 30)])
          epoch_powers.append(band_power)
      band_powers.append(epoch_powers)

  band_powers = np.array(band_powers)
  return band_powers

In [10]:
band_power = calculate_band_power(data,sfreq)

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/scipy/signal/_spectral_py.py:2014: UserWarning: nperseg = 500 is greater than input length  = 201, using nperseg = 201
  warnings.warn('nperseg = {0:d} is greater than input length '


In [11]:
band_power.shape

(288, 22)

Band power shape is EVENT X CHANNEL

### Edge Features

Constructing Connectivity matrix using correlation

In [12]:
def compute_connectivity(data):
    """
    Compute the Pearson correlation between each pair of channels for a single event.
    Parameters:
    data (numpy array): The input EEG data of shape (channels, datapoints).
    Returns:
    connectivity_matrix (numpy array): A matrix of shape (channels, channels) representing connectivity.
    """
    return np.corrcoef(data)  # Returns a (channels x channels) matrix

def create_graph_from_connectivity(connectivity_matrix, band_power, threshold=0.3):
    """
    Create a graph from the connectivity matrix and band powers.
    Parameters:
    connectivity_matrix (numpy array): The connectivity matrix of shape (channels, channels).
    band_power (numpy array): Band power features for each channel (shape: [channels]).
    threshold (float): The threshold for filtering weak connections.
    Returns:
    Data: A PyTorch Geometric graph data object.
    """
    num_channels = connectivity_matrix.shape[0]
    
    # Node features are the band powers for each channel
    node_features = torch.tensor(band_power, dtype=torch.float).unsqueeze(1)  # Shape: [num_channels, 1]
    
    # Create edges based on threshold
    edge_index = []
    edge_attr = []
    
    for i in range(num_channels):
        for j in range(i+1, num_channels):
            if abs(connectivity_matrix[i, j]) > threshold:
                edge_index.append([i, j])
                edge_index.append([j, i])
                edge_attr.append(connectivity_matrix[i, j])
                edge_attr.append(connectivity_matrix[i, j])
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr)


# Call the updated function

In [13]:
def create_graphs_for_all_events(dataset, labels, band_power, threshold=0.3):
    """
    Create a graph for each event (epoch) in the dataset.
    Parameters:
    dataset (numpy array): EEG data of shape (events, channels, datapoints).
    labels (numpy array): Labels for each event (epoch).
    band_power (numpy array): Precomputed band power features of shape (events, channels).
    threshold (float): Threshold for filtering weak connections in the connectivity matrix.
    Returns:
    all_graphs (list): A list of graph objects for each event.
    """
    all_graphs = []
    for i, event in enumerate(dataset):
        # Compute connectivity matrix for this event (correlation between channels)
        connectivity_matrix = compute_connectivity(event)
        
        # Create a graph for this event using its connectivity and band power
        graph_data = create_graph_from_connectivity(connectivity_matrix, band_power[i], threshold)
        graph_data.y = torch.tensor([labels[i]], dtype=torch.long)  # Add label for classification
        
        all_graphs.append(graph_data)
    return all_graphs

In [ ]:
graph_list = create_graphs_for_all_events(data, labels, band_power)
graph_list

## MODEL TRAINING

In [15]:
class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super(GCN, self).__init__()
        # GCN branch
        self.gcn1 = SAGEConv(num_node_features, hidden_channels )
        self.gcn2 = SAGEConv(hidden_channels, 50)
        # GAT branch
        self.gat1 = GATConv(num_node_features, hidden_channels)
        self.gat2 = GATConv(hidden_channels, 50)
        # Chebyshev branch
        self.cheb1 = ChebConv(num_node_features, hidden_channels, K=2)
        self.cheb2 = ChebConv(hidden_channels, 50, K=2)
        # GraphSAGE branch
        self.sage1 = SAGEConv(num_node_features, hidden_channels)
        self.sage2 = SAGEConv(hidden_channels, 50)
        # Fully connected layer for classification
        self.fc = torch.nn.Linear(50 * 4, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # x = x.view(-1, 1)        
        x_gcn = F.relu(self.gcn1(x, edge_index))
        x_gcn = F.dropout(x_gcn, training=self.training)
        x_gcn = F.relu(self.gcn2(x_gcn, edge_index))

        # GAT branch
        x_gat = F.relu(self.gat1(x, edge_index))
        x_gat = F.dropout(x_gat, training=self.training)
        x_gat = F.relu(self.gat2(x_gat, edge_index))

        # Chebyshev branch
        x_cheb = F.relu(self.cheb1(x, edge_index))
        x_cheb = F.dropout(x_cheb, training=self.training)
        x_cheb = F.relu(self.cheb2(x_cheb, edge_index))

        # GraphSAGE branch
        x_sage = F.relu(self.sage1(x, edge_index))
        x_sage = F.dropout(x_sage, training=self.training)
        x_sage = F.relu(self.sage2(x_sage, edge_index))

        # Concatenate the outputs from all branches
        x = torch.cat((x_gcn, x_gat, x_cheb, x_sage), dim=1)

        # Apply the pooling layer to get graph-level representation
        x = torch_geometric.nn.global_mean_pool(x, data.batch)

        # Apply the classification layer
        x = self.fc(x)

        return F.log_softmax(x, dim=1)

In [16]:
import torch_geometric
hidden_channels = 100  # Number of hidden units
num_node_features = 1  # This is the sequence length
num_classes = 4  # Assuming binary classification, adjust based on your dataset

# Initialize the model
model = GCN(num_node_features, hidden_channels, num_classes)
print(model)

GCN(
  (gcn1): SAGEConv(1, 100, aggr=mean)
  (gcn2): SAGEConv(100, 50, aggr=mean)
  (gat1): GATConv(1, 100, heads=1)
  (gat2): GATConv(100, 50, heads=1)
  (cheb1): ChebConv(1, 100, K=2, normalization=sym)
  (cheb2): ChebConv(100, 50, K=2, normalization=sym)
  (sage1): SAGEConv(1, 100, aggr=mean)
  (sage2): SAGEConv(100, 50, aggr=mean)
  (fc): Linear(in_features=200, out_features=4, bias=True)
)


In [17]:
train_indices, test_indices = train_test_split(range(len(labels)), test_size=0.2, random_state=42)

In [18]:
train_dataset = create_graphs_for_all_events(data[train_indices], labels[train_indices],band_power[train_indices])
test_dataset = create_graphs_for_all_events(data[test_indices], labels[test_indices],band_power[train_indices])

In [19]:
test_dataset[0]

Data(x=[22, 1], edge_index=[2, 330], edge_attr=[330], y=[1])

In [ ]:
import torch.optim as optim  # Import torch.optim for the optimizer


def train_model(train_dataset, test_dataset, model, num_epochs=25, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Train loop
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()

            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = out.max(dim=1)
            correct += (predicted == data.y).sum().item()
            total += data.y.size(0)

        train_accuracy = correct / total
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}, Training Accuracy: {train_accuracy:.4f}')
        
    # Evaluate on the test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data)
            _, predicted = out.max(dim=1)
            correct += (predicted == data.y).sum().item()
            total += data.y.size(0)
            
    test_accuracy = correct / total
    print(f'Test Accuracy: {test_accuracy:.4f}')
    return model, test_accuracy

model = GCN(num_node_features=num_node_features, hidden_channels=hidden_channels, num_classes=num_classes)
trained_model, accuracy = train_model(train_dataset, test_dataset, model)

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv

In [15]:
class GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.5, model_type="GCN"):
        super(GNN, self).__init__()
        self.convs = nn.ModuleList()
        if model_type == "GCN":
            self.convs.append(GCNConv(input_dim, hidden_dim))
        elif model_type == "GAT":
            self.convs.append(GATConv(input_dim, hidden_dim, heads=8))
        elif model_type == "GraphSAGE":
            self.convs.append(SAGEConv(input_dim, hidden_dim))
        else:
            raise ValueError("Invalid model type")
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))  # Replace with GATConv or SAGEConv if needed
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.fc(x)
        return x

In [16]:
from torch_geometric.data import DataLoader
dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN(1, 50, 4, 2).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()  # Replace with appropriate loss function for your task

for epoch in range(num_epochs):
    model.train()
    for data in dataloader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

In [86]:
train_data, train_labels, sfreq= read_data('/Users/siddharth/Downloads/BCICIV_2a_gdf/A01T.gdf')

Extracting EDF parameters from /Users/siddharth/Downloads/BCICIV_2a_gdf/A01T.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/Users/siddharth/opt/anaconda3/envs/Brainconnectivity/lib/python3.9/contextlib.py:126: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 sec)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.1s remaining:    0.0s


Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '769', '770', '771', '772']
Not setting metadata
288 matching events found
Setting baseline interval to [-0.1, 0.0] sec
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 288 events and 201 original time points ...
0 bad epochs dropped


[Parallel(n_jobs=1)]: Done  22 out of  22 | elapsed:    0.5s finished


In [88]:
def calculate_band_power(data,sfreq):
  band_powers = []
  
  for epoch in data:  # Loop over each epoch
      epoch_powers = []
      for channel_data in epoch:  # Loop over each channel in the epoch
          freqs, psd = welch(channel_data, sfreq, nperseg=sfreq*2)  # Compute the PSD
          # Compute the band power by integrating the power spectral density (PSD) in the 8-30 Hz band
          band_power = np.trapz(psd[(freqs >= 8) & (freqs <= 30)], freqs[(freqs >= 8) & (freqs <= 30)])
          epoch_powers.append(band_power)
      band_powers.append(epoch_powers)

  band_powers = np.array(band_powers)
  return band_powers

In [89]:
train_band_power = calculate_band_power(train_data,sfreq)

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/scipy/signal/_spectral_py.py:2014: UserWarning: nperseg = 500 is greater than input length  = 201, using nperseg = 201
  warnings.warn('nperseg = {0:d} is greater than input length '


In [90]:
test_data, test_labels, sfreq = read_data('/Users/siddharth/Downloads/BCICIV_2a_gdf/A01E.gdf')
test_band_power = calculate_band_power(test_data,sfreq)

Extracting EDF parameters from /Users/siddharth/Downloads/BCICIV_2a_gdf/A01E.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 686999  =      0.000 ...  2747.996 secs...


/Users/siddharth/opt/anaconda3/envs/Brainconnectivity/lib/python3.9/contextlib.py:126: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 sec)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.2s remaining:    0.0s


Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '783']
Not setting metadata
288 matching events found
Setting baseline interval to [-0.1, 0.0] sec
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 288 events and 201 original time points ...
0 bad epochs dropped


[Parallel(n_jobs=1)]: Done  22 out of  22 | elapsed:    0.8s finished
/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_94851/3686818143.py:8: RuntimeWarning: No matching events found for 8 (event id 8)
  epoch = mne.Epochs(raw,events[0],event_id=[7,8,9,10],tmin = -0.1,tmax=0.7,on_missing='warn')
/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_94851/3686818143.py:8: RuntimeWarning: No matching events found for 9 (event id 9)
  epoch = mne.Epochs(raw,events[0],event_id=[7,8,9,10],tmin = -0.1,tmax=0.7,on_missing='warn')
/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_94851/3686818143.py:8: RuntimeWarning: No matching events found for 10 (event id 10)
  epoch = mne.Epochs(raw,events[0],event_id=[7,8,9,10],tmin = -0.1,tmax=0.7,on_missing='warn')
/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/scipy/signal/_spectral_py.py:2014: UserWarning: nperseg = 500 is greater than input length  = 201, using nperseg = 201
  warnings.warn('nperseg = {0:d} is gre

In [76]:
import scipy.stats as stats
def adjency_matrix(data):
    correlation_matrices = []

    for event in range(data.shape[0]):
        event_data = data[event]
        
        corr_matrix = np.corrcoef(event_data)
        
        correlation_matrices.append(corr_matrix)

    correlation_matrices = np.array(correlation_matrices)
    correlation1 = correlation_matrices[labels == 7]
    correlation2 = correlation_matrices[labels == 8]
    correlation3 = correlation_matrices[labels == 9]
    correlation4 = correlation_matrices[labels == 10]
    t2 = stats.ttest_ind(correlation1[:,0,0], correlation2[:,0,0])
    connectivity_matrix = np.zeros((22, 22))
    alpha = 0.05
    for i in range(22):
        for j in range(22):
            values1 = correlation1[:, i, j]
            values2 = correlation2[:, i, j]
            t_stat, p_value = stats.ttest_ind(values1, values2)
            if p_value > alpha:
                connectivity_matrix[i, j] = 1

    return connectivity_matrix

In [46]:
# connectivity_matrix.shape (22,22) for 1 subject

(22, 22)

In [117]:
import scipy.io
mat = scipy.io.loadmat('/Users/siddharth/Downloads/true_labels/A01E.mat')
mat = np.array(mat['classlabel']).flatten()
mat = mat - 1
mat = mat[(mat == 0)| (mat == 1)]
mat.shape

(144,)

In [94]:
def create_graph(data,labels,band_power):
    connectivity_matrix = adjency_matrix(data)
    edge_index = torch.tensor(np.array(np.nonzero(connectivity_matrix)), dtype=torch.long)
    graph_data_list = []
    for event_idx in range(band_power.shape[0]):
        node_features = torch.tensor(band_power[event_idx], dtype=torch.float).view(-1, 1)  # (22, 1)
        graph_data = Data(x=node_features, edge_index=edge_index)
        graph_data.y = torch.tensor([labels[event_idx]], dtype=torch.long) 
        graph_data_list.append(graph_data)
    return graph_data_list

In [95]:
train_dataset = create_graph(train_data, train_labels,train_band_power)
test_dataset = create_graph(test_data, test_labels,test_band_power)

/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_94851/163099985.py:17: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  t2 = stats.ttest_ind(correlation1[:,0,0], correlation2[:,0,0])
/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_94851/163099985.py:24: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  t_stat, p_value = stats.ttest_ind(values1, values2)


In [99]:
class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super(GCN, self).__init__()
        # GCN branch
        self.gcn1 = SAGEConv(num_node_features, hidden_channels )
        self.gcn2 = SAGEConv(hidden_channels, 50)
        # GAT branch
        self.gat1 = GATConv(num_node_features, hidden_channels)
        self.gat2 = GATConv(hidden_channels, 50)
        # Chebyshev branch
        self.cheb1 = ChebConv(num_node_features, hidden_channels, K=2)
        self.cheb2 = ChebConv(hidden_channels, 50, K=2)
        # GraphSAGE branch
        self.sage1 = SAGEConv(num_node_features, hidden_channels)
        self.sage2 = SAGEConv(hidden_channels, 50)
        # Fully connected layer for classification
        self.fc = torch.nn.Linear(50 * 4, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # x = x.view(-1, 1)        
        x_gcn = F.relu(self.gcn1(x, edge_index))
        x_gcn = F.dropout(x_gcn, training=self.training)
        x_gcn = F.relu(self.gcn2(x_gcn, edge_index))

        # GAT branch
        x_gat = F.relu(self.gat1(x, edge_index))
        x_gat = F.dropout(x_gat, training=self.training)
        x_gat = F.relu(self.gat2(x_gat, edge_index))

        # Chebyshev branch
        x_cheb = F.relu(self.cheb1(x, edge_index))
        x_cheb = F.dropout(x_cheb, training=self.training)
        x_cheb = F.relu(self.cheb2(x_cheb, edge_index))

        # GraphSAGE branch
        x_sage = F.relu(self.sage1(x, edge_index))
        x_sage = F.dropout(x_sage, training=self.training)
        x_sage = F.relu(self.sage2(x_sage, edge_index))

        # Concatenate the outputs from all branches
        x = torch.cat((x_gcn, x_gat, x_cheb, x_sage), dim=1)

        # Apply the pooling layer to get graph-level representation
        x = torch_geometric.nn.global_mean_pool(x, data.batch)

        # Apply the classification layer
        x = self.fc(x)

        return F.log_softmax(x, dim=1)

In [100]:
import torch_geometric
hidden_channels = 100  # Number of hidden units
num_node_features = 1  # This is the sequence length
num_classes = 4  # Assuming binary classification, adjust based on your dataset

# Initialize the model
model = GCN(num_node_features, hidden_channels, num_classes)
print(model)

GCN(
  (gcn1): SAGEConv(1, 100, aggr=mean)
  (gcn2): SAGEConv(100, 50, aggr=mean)
  (gat1): GATConv(1, 100, heads=1)
  (gat2): GATConv(100, 50, heads=1)
  (cheb1): ChebConv(1, 100, K=2, normalization=sym)
  (cheb2): ChebConv(100, 50, K=2, normalization=sym)
  (sage1): SAGEConv(1, 100, aggr=mean)
  (sage2): SAGEConv(100, 50, aggr=mean)
  (fc): Linear(in_features=200, out_features=4, bias=True)
)


In [101]:
import torch.optim as optim  # Import torch.optim for the optimizer


def train_model(train_dataset, test_dataset, model, num_epochs=25, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Train loop
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()

            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = out.max(dim=1)
            correct += (predicted == data.y).sum().item()
            total += data.y.size(0)

        train_accuracy = correct / total
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}, Training Accuracy: {train_accuracy:.4f}')
        
    # Evaluate on the test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data)
            _, predicted = out.max(dim=1)
            correct += (predicted == data.y).sum().item()
            total += data.y.size(0)
            
    test_accuracy = correct / total
    print(f'Test Accuracy: {test_accuracy:.4f}')
    return model, test_accuracy

trained_model, accuracy = train_model(train_dataset, test_dataset, model)

RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xe

RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xe

RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xe

SystemError: initialization of _pywrap_checkpoint_reader raised unreported exception

In [7]:
from mne.time_frequency import psd_multitaper

ImportError: cannot import name 'psd_multitaper' from 'mne.time_frequency' (/Users/siddharth/opt/anaconda3/envs/Brainconnectivity/lib/python3.9/site-packages/mne/time_frequency/__init__.py)

In [5]:
# import mne.time_frequency.psd_multitaper as psd
focused_bands = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'low_gamma': (30, 50)
}

def compute_band_power(psd, freqs, band):
    band_freqs = (freqs >= band[0]) & (freqs <= band[1])
    return np.sum(psd[:, band_freqs], axis=1)

In [14]:
def extract_features(event_data):
    # Calculate PSD for each channel in an event
    psd, freqs = mne.time_frequency.psd_multitaper(
        event_data, 
        fmin=0.5, 
        fmax=50, 
        adaptive=True, 
        normalization='full', 
        verbose=False
    )
    node_features = []
    for i in range(data[1]):
        # Calculate power in each band
        band_powers = [compute_band_power(psd[i:i+1, :], freqs, band) for band in focused_bands.values()]
        total_power = np.sum(psd[i, :])
        band_percentages = [power / total_power for power in band_powers]
        # Feature vector: [delta%, theta%, alpha%, beta%, low_gamma%, total_power, location_code]
        node_features.append(band_percentages + [total_power, i])
    return torch.tensor(node_features, dtype=torch.float)

In [16]:
feature = extract_features(data[0])

AttributeError: 'numpy.ndarray' object has no attribute 'compute_psd'

In [10]:
data.compute_psd()

AttributeError: 'numpy.ndarray' object has no attribute 'compute_psd'

In [6]:
psd, freqs = mne.time_frequency.psd_multitaper(data[0], fmin=0.5, fmax=50, adaptive=True, normalization='full', verbose=False)

AttributeError: No mne.time_frequency attribute psd_multitaper

In [13]:
import mne
mne.sys_info()

Platform             macOS-15.0-arm64-arm-64bit
Python               3.9.19 | packaged by conda-forge | (main, Mar 20 2024, 12:55:20)  [Clang 16.0.6 ]
Executable           /Users/siddharth/opt/anaconda3/envs/Brainconnectivity/bin/python
CPU                  arm (8 cores)
Memory               8.0 GB



AttributeError: 'NoneType' object has no attribute 'split'

In [14]:
raw = mne.io.read_raw_gdf('/Users/siddharth/Downloads/BCICIV_2a_gdf/A01T.gdf',eog=['EOG-left', 'EOG-central', 'EOG-right'],preload=True)
raw.drop_channels(['EOG-left', 'EOG-central', 'EOG-right'])
raw.set_eeg_reference()
raw.filter(l_freq=8., h_freq=30.)
events = mne.events_from_annotations(raw)
epoch = mne.Epochs(raw,events[0],event_id=[7,8,9,10],tmin = -0.1,tmax=0.7,on_missing='warn')

Extracting EDF parameters from /Users/siddharth/Downloads/BCICIV_2a_gdf/A01T.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/Users/siddharth/opt/anaconda3/envs/Brainconnectivity/lib/python3.9/contextlib.py:126: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 s)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.1s remaining:    0.0s


Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '769', '770', '771', '772']
Not setting metadata
288 matching events found
Setting baseline interval to [-0.1, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated


[Parallel(n_jobs=1)]: Done  22 out of  22 | elapsed:    0.3s finished


In [18]:
features = epoch.compute_psd(method="multitaper", tmin=-0.1, tmax=0.7, fmin=5, fmax=30, picks="eeg")

Using data from preloaded Raw for 288 events and 201 original time points ...
    Using multitaper spectrum estimation with 7 DPSS windows


In [19]:
features.shape

(288, 22, 20)